# 📈 Sales Prediction Using Python

This notebook builds and compares two machine learning models — **Linear Regression** and **Random Forest Regressor** — to predict product **Sales** based on advertising spend across three channels: **TV**, **Radio**, and **Newspaper**.

**Dataset:** `Advertising.csv` — 200 records of advertising budgets (in thousands of dollars) and the resulting sales (in thousands of units).

**Workflow:**
1. Load & clean the data
2. Explore the data (EDA)
3. Check correlations between features and sales
4. Split data into train/test sets
5. Train Linear Regression and Random Forest models
6. Evaluate & compare model performance
7. Analyze residuals and feature importance
8. Predict sales for a new advertising budget

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load & Clean the Data

In [2]:
df = pd.read_csv("Advertising.csv")
df.head()

,Unnamed: 0,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


Drop the unnamed index column that came from the CSV export.

In [3]:
df = df.drop("Unnamed: 0", axis=1)
df.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


In [4]:
df.shape

(200, 4)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   TV         200 non-null    float64
 1   Radio      200 non-null    float64
 2   Newspaper  200 non-null    float64
 3   Sales      200 non-null    float64
dtypes: float64(4)
memory usage: 6.4 KB


In [6]:
df.describe().round().T

,count,mean,std,min,25%,50%,75%,max
TV,200.0,147.0,86.0,1.0,74.0,150.0,219.0,296.0
Radio,200.0,23.0,15.0,0.0,10.0,23.0,37.0,50.0
Newspaper,200.0,31.0,22.0,0.0,13.0,26.0,45.0,114.0
Sales,200.0,14.0,5.0,2.0,10.0,13.0,17.0,27.0


Check for missing values and duplicate rows before moving on.

In [7]:
df.isnull().sum()

TV           0
Radio        0
Newspaper    0
Sales        0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df["Sales"].value_counts()

Sales
9.7     5
11.7    4
12.9    4
15.9    4
20.7    3
       ..
17.0    1
18.3    1
22.3    1
14.0    1
25.5    1
Name: count, Length: 121, dtype: int64

## 3. Exploratory Data Analysis (EDA)

### 3.1 Relationship Between Each Advertising Channel and Sales

In [10]:
fig = px.scatter(
    df,
    x="TV",
    y="Sales",
    title="TV Advertising vs Sales"
)

fig.show()

In [11]:
fig = px.scatter(
    df,
    x="Radio",
    y="Sales",
    title="Radio Advertising vs Sales"
)

fig.show()

In [12]:
fig = px.scatter(
    df,
    x="Newspaper",
    y="Sales",
    title="Newspaper Advertising vs Sales"
)

fig.show()

### 3.2 Distributions

In [13]:
fig = px.histogram(
    df,
    x="Sales",
    title="Sales Distribution"
)

fig.show()

In [14]:
fig = px.box(
    df,
    y=["TV", "Radio", "Newspaper", "Sales"],
    title="Distribution and Outliers of All Variables"
)

fig.show()

In [15]:
fig = px.histogram(
    df,
    x=["TV", "Radio", "Newspaper"],
    barmode="overlay",
    title="Advertising Spend Distribution"
)

fig.show()

## 4. Correlation Analysis

In [16]:
correlation = df.corr()
correlation

,TV,Radio,Newspaper,Sales
TV,1.000000,0.054809,0.056648,0.782224
Radio,0.054809,1.000000,0.354104,0.576223
Newspaper,0.056648,0.354104,1.000000,0.228299
Sales,0.782224,0.576223,0.228299,1.000000


In [17]:
fig = px.imshow(
    correlation,
    text_auto=True,
    title="Correlation Heatmap"
)

fig.show()

**Observation:** `TV` has the strongest correlation with `Sales`, followed by `Radio`. `Newspaper` shows the weakest relationship.

## 5. Train/Test Split

In [18]:
X = df.drop("Sales", axis=1)
y = df["Sales"]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (160, 3)
X_test: (40, 3)
y_train: (160,)
y_test: (40,)


## 6. Model Training

We train two models and compare them: a simple, interpretable **Linear Regression** baseline, and a more flexible **Random Forest Regressor**.

In [21]:
linear_model = LinearRegression()

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

linear_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
linear_pred = linear_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

In [23]:
print("Linear Regression Predictions:")
print(linear_pred[:10])

print("\nRandom Forest Predictions:")
print(rf_pred[:10])

Linear Regression Predictions:
[16.4080242  20.88988209 21.55384318 10.60850256 22.11237326 13.10559172
 21.05719192  7.46101034 13.60634581 15.15506967]

Random Forest Predictions:
[17.698 21.804 20.628  6.793 22.927 13.379 22.376  9.688 11.826 15.54 ]


## 7. Model Evaluation

In [24]:
linear_mae = mean_absolute_error(y_test, linear_pred)
linear_mse = mean_squared_error(y_test, linear_pred)
linear_rmse = np.sqrt(linear_mse)
linear_r2 = r2_score(y_test, linear_pred)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("Linear Regression")
print("MAE:", linear_mae)
print("MSE:", linear_mse)
print("RMSE:", linear_rmse)
print("R² Score:", linear_r2)

print("\nRandom Forest")
print("MAE:", rf_mae)
print("MSE:", rf_mse)
print("RMSE:", rf_rmse)
print("R² Score:", rf_r2)

Linear Regression
MAE: 1.4607567168117606
MSE: 3.1740973539761055
RMSE: 1.7815996615334506
R² Score: 0.899438024100912

Random Forest
MAE: 0.6200999999999988
MSE: 0.5907322499999988
RMSE: 0.7685910811348248
R² Score: 0.9812843792541843


In [25]:
model_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [linear_mae, rf_mae],
    "MSE": [linear_mse, rf_mse],
    "RMSE": [linear_rmse, rf_rmse],
    "R² Score": [linear_r2, rf_r2]
})

model_comparison

,Model,MAE,MSE,RMSE,R² Score
0,Linear Regression,1.460757,3.174097,1.781600,0.899438
1,Random Forest,0.620100,0.590732,0.768591,0.981284


In [26]:
fig = px.bar(
    model_comparison,
    x="Model",
    y=["MAE", "RMSE", "R² Score"],
    barmode="group",
    title="Linear Regression vs Random Forest"
)

fig.show()

## 8. Actual vs Predicted Sales

In [27]:
comparison = pd.DataFrame({
    "Actual Sales": y_test.values,
    "Linear Regression": linear_pred,
    "Random Forest": rf_pred
})

comparison.head(10)

,Actual Sales,Linear Regression,Random Forest
0,16.9,16.408024,17.698
1,22.4,20.889882,21.804
2,21.4,21.553843,20.628
3,7.3,10.608503,6.793
4,24.7,22.112373,22.927
5,12.6,13.105592,13.379
6,22.3,21.057192,22.376
7,8.4,7.461010,9.688
8,11.5,13.606346,11.826
9,14.9,15.155070,15.540


In [28]:
fig = px.scatter(
    comparison,
    x="Actual Sales",
    y="Linear Regression",
    title="Actual vs Predicted Sales - Linear Regression"
)

fig.show()

In [29]:
fig = px.scatter(
    comparison,
    x="Actual Sales",
    y="Random Forest",
    title="Actual vs Predicted Sales - Random Forest"
)

fig.show()

## 9. Residual Analysis

Residuals should scatter randomly around zero with no clear pattern — a pattern would suggest the model is missing structure in the data.

In [30]:
comparison["Linear Error"] = comparison["Actual Sales"] - comparison["Linear Regression"]
comparison["Random Forest Error"] = comparison["Actual Sales"] - comparison["Random Forest"]

comparison.head(10)

,Actual Sales,Linear Regression,Random Forest,Linear Error,Random Forest Error
0,16.9,16.408024,17.698,0.491976,-0.798
1,22.4,20.889882,21.804,1.510118,0.596
2,21.4,21.553843,20.628,-0.153843,0.772
3,7.3,10.608503,6.793,-3.308503,0.507
4,24.7,22.112373,22.927,2.587627,1.773
5,12.6,13.105592,13.379,-0.505592,-0.779
6,22.3,21.057192,22.376,1.242808,-0.076
7,8.4,7.461010,9.688,0.938990,-1.288
8,11.5,13.606346,11.826,-2.106346,-0.326
9,14.9,15.155070,15.540,-0.255070,-0.640


In [31]:
fig = px.scatter(
    comparison,
    x="Linear Regression",
    y="Linear Error",
    title="Residual Analysis - Linear Regression",
    labels={
        "Linear Regression": "Predicted Sales",
        "Linear Error": "Residual Error"
    }
)

fig.add_hline(y=0)

fig.show()

In [32]:
fig = px.scatter(
    comparison,
    x="Random Forest",
    y="Random Forest Error",
    title="Residual Analysis - Random Forest",
    labels={
        "Random Forest": "Predicted Sales",
        "Random Forest Error": "Residual Error"
    }
)

fig.add_hline(y=0)

fig.show()

## 10. Feature Importance

In [33]:
linear_coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": linear_model.coef_
})

linear_coefficients

,Feature,Coefficient
0,TV,0.044730
1,Radio,0.189195
2,Newspaper,0.002761


In [34]:
rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

rf_importance = rf_importance.sort_values(
    by="Importance",
    ascending=False
)

rf_importance

,Feature,Importance
0,TV,0.624810
1,Radio,0.362201
2,Newspaper,0.012989


In [35]:
fig = px.bar(
    rf_importance,
    x="Feature",
    y="Importance",
    title="Random Forest Feature Importance"
)

fig.show()

## 11. Predicting Sales for a New Advertising Budget

Example: what sales would we expect for a budget of **$150K on TV, $25K on Radio, and $20K on Newspaper**?

In [36]:
new_data = pd.DataFrame({
    "TV": [150],
    "Radio": [25],
    "Newspaper": [20]
})

linear_prediction = linear_model.predict(new_data)
rf_prediction = rf_model.predict(new_data)

print("Linear Regression Prediction:", linear_prediction[0])
print("Random Forest Prediction:", rf_prediction[0])

Linear Regression Prediction: 14.473593601116834
Random Forest Prediction: 15.15099999999999


## 12. Final Comparison & Conclusion

In [37]:
final_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [linear_mae, rf_mae],
    "RMSE": [linear_rmse, rf_rmse],
    "R² Score": [linear_r2, rf_r2]
})

final_comparison

,Model,MAE,RMSE,R² Score
0,Linear Regression,1.460757,1.781600,0.899438
1,Random Forest,0.620100,0.768591,0.981284


In [38]:
if rf_r2 > linear_r2:
    best_model = "Random Forest"
else:
    best_model = "Linear Regression"

print("Better Model:", best_model)

Better Model: Random Forest


### Conclusion

- **TV advertising** has the strongest impact on sales, followed by **Radio**, while **Newspaper** contributes the least.
- The **Random Forest Regressor** captures the relationship between advertising spend and sales more accurately than **Linear Regression**, achieving a lower error and a higher R² score on the test set.
- This makes Random Forest the better choice for predicting sales from advertising budgets in this dataset, though Linear Regression remains useful for its simplicity and interpretability (the coefficients directly show each channel's effect on sales).